### Import Dependencies

In [ ]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

### Download an example reference data point from LangSmith

In [ ]:
from dotenv import load_dotenv

load_dotenv("../../.env")

In [ ]:
ls_client = Client()

### Build Eval Dataset

In [ ]:
coordinator_eval_dataset = [
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "What is the weather today?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "",
                "final_answer": True
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can I get some earphones?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "product_qna_agent",
                "final_answer": False
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you add an item with ID B09NLTDHQ6 to my cart?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "shopping_cart_agent",
                "final_answer": False
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you add those earphones to my cart?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "",
                "final_answer": True
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you add the best items to my cart? I am looking for laptop bags."}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "product_qna_agent",
                "final_answer": False
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you find some good reviews for items in my cart?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "shopping_cart_agent",
                "final_answer": False
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you put the items with the most positive user reviews to my cart?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "product_qna_agent",
                "final_answer": False
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "What kind of stuff do you sell?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "",
                "final_answer": True
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you help me with my order?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "",
                "final_answer": True
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you add two, ideally red tablets to my cart?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "product_qna_agent",
                "final_answer": False
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you reserve a red leather laptop bag for me?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "product_qna_agent",
                "final_answer": False
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you reserve my shopping cart?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "shopping_cart_agent",
                "final_answer": False
            }
        }
    }
]

### Upload the dataset to LangSmith

In [ ]:
dataset_name = "coordinator-first-delegation-evaluation"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="This dataset contains the first delegation of the coordinator agent."
)

In [ ]:
for item in coordinator_eval_dataset:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs=item["inputs"],
        outputs=item["outputs"]
    )

### Coordinator Agent Evaluation

In [ ]:
from pydantic import BaseModel

from langchain_openai import ChatOpenAI

from langsmith import traceable

from langchain_core.messages import SystemMessage, AIMessage
from IPython.display import display

from typing import Any, Annotated, List
from pydantic import Field
from operator import add

from jinja2 import Template
from utils.utils import postprocess_response

In [ ]:
MAX_ITERATIONS_PRODUCT_QNA = 5
MAX_ITERATIONS_SHOPPING_CART = 4
MAX_ITERATIONS_WAREHOUSE_MANAGER = 4
MAX_ITERATIONS_COORDINATOR = 6

In [ ]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [ ]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available agents and skills
 
### `product_qna_agent`
 
Owns all product knowledge: catalog, specs, pricing, reviews, recommendations. This is the default agent for any question about what exists or what to buy.
 
Skills:
 
- `get_formatted_item_context` — Search available products and return the top k matching inventory items.
- `get_formatted_reviews_context` — Get the top k reviews matching a query for a list of prefiltered items.
Does not: modify the cart, check warehouse stock, or reserve anything.
 
### `shopping_cart_agent`
 
Owns the state of the user's shopping cart.
 
Skills:
 
- `add_to_shopping_cart` — Add a list of provided items to the shopping cart.
- `get_shopping_cart` — Retrieve all items in a user's shopping cart.
- `remove_from_cart` —  Remove an item completely from the shopping cart.
Does not: recommend products, check warehouse stock, reserve stock, or place orders.
 
### `warehouse_manager_agent`
 
Owns warehouse inventory and reservations.
 
Skills:
 
- `check_warehouse_availability` — Check availability of items across warehouses, including partial fulfillment options.
- `reserve_warehouse_items` — Reserve items from multiple warehouses in a single transaction.
Does not: recommend products, modify the cart.
"""

    template = Template(prompt_template)

    prompt = template.render()

    if state.coordinator_agent.iteration == MAX_ITERATIONS_COORDINATOR:
        tools = [FinalAgentResponse]
    else:
        tools = [Plan, FinalAgentResponse]

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        tools,
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    next_agent = ""
    next_agent_task = ""

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            next_agent_task = response.tool_calls[0].get("args").get("next_agent_task")
            response = AIMessage(content=f"[coordinator_agent decision] Next agent: {next_agent}. Next agent task: {next_agent_task}")
        else:
            postprocessed_response = postprocess_response(response, "FinalAgentResponse")

            final_answer = postprocessed_response.get("final_answer")
            answer = postprocessed_response.get("answer")
            response = postprocessed_response.get("response")

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "next_agent": next_agent,
            "next_agent_task": next_agent_task
        },
        "answer": answer
    }

### Run against LangSmith

In [ ]:
def evaluate_coordinator_delegation(run, example):

    final_answer_match = run.outputs["coordinator_agent"]["final_answer"] == example.outputs["coordinator_agent"]["final_answer"]
    next_agent_match = run.outputs["coordinator_agent"]["next_agent"] == example.outputs["coordinator_agent"]["next_agent"]

    return final_answer_match and next_agent_match

In [ ]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-first-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=6,
    num_repetitions=1
)

### Extract Evaluation Results

In [ ]:
results

In [ ]:
results.experiment_name

In [ ]:
results_resp = ls_client.read_project(
    project_name=results.experiment_name,
    include_stats=True
)

In [ ]:
results_resp

In [ ]:
results_resp.feedback_stats

In [ ]:
results_resp.feedback_stats.get("evaluate_coordinator_delegation")

In [ ]:
results_resp.feedback_stats.get("evaluate_coordinator_delegation").get("avg")

In [ ]:
results_resp.feedback_stats.get("evaluate_coordinator_delegation").get("errors")